In [1]:
import pandas as pd

x = pd.read_csv('data/x.csv')
r = pd.read_csv('data/reddit.csv')


In [3]:
print(x.columns)
print(r.columns)

Index(['tweet_id', 'text', 'user_id', 'username', 'user_name', 'user_bio',
       'followers_count', 'user_location', 'statuses_count', 'profile_url',
       'created_at', 'retweet_count', 'favorite_count', 'reply_count',
       'is_reply', 'tweet_url', 'search_phrase', 'matched_phrase',
       'matched_keyword', 'match_distance', 'diagnosis', 'medications_found',
       '_source_file', 'condition_ar', 'qwen3-235b-a22b-instruct-2507',
       'qwen3-235b-a22b-instruct-2507__label',
       'qwen3-235b-a22b-instruct-2507__reasons',
       'qwen3-235b-a22b-instruct-2507__status', 'gpt-4.1', 'gpt-4.1__label',
       'gpt-4.1__reasons', 'gpt-4.1__status', 'Jais-2-8B-Chat',
       'Jais-2-8B-Chat__label', 'Jais-2-8B-Chat__reasons',
       'Jais-2-8B-Chat__status', 'genuine_votes', 'majority_genuine',
       'unanimous_genuine'],
      dtype='str')
Index(['Unnamed: 0', 'Post_ID', 'Comment_ID', 'Content_Type', 'Author',
       'Subreddit', 'Created_UTC', 'Year_Month', 'Score', 'Num_Comments',
 

In [4]:
r[r['genuine_votes'] == True]['Author'].nunique()

4515

In [5]:
import pandas as pd
from pathlib import Path

Path("data/metadata").mkdir(parents=True, exist_ok=True)

x = x.copy()
r = r.copy()
x.columns = x.columns.str.strip().str.lower()
r.columns = r.columns.str.strip().str.lower()

x_df = x[['tweet_id', 'text', 'username', 'created_at', 'diagnosis']].rename(columns={
    'tweet_id': 'id',
    'username': 'author',
    'created_at': 'created_utc'
})

x_df['subreddit'] = pd.NA
x_df['is_comment'] = pd.NA
x_df['platform'] = 'twitter'

r['is_comment'] = r['content_type'].fillna('').astype(str).str.lower().eq('comment')

r_df = r[['post_id', 'text', 'author', 'created_utc', 'subreddit', 'is_comment', 'diagnosis']].rename(columns={
    'post_id': 'id'
})

r_df['platform'] = 'reddit'

cols = ['id', 'text', 'author', 'created_utc', 'subreddit', 'is_comment', 'diagnosis', 'platform']

x_df['created_utc'] = pd.to_datetime(x_df['created_utc'], utc=True, errors='coerce')
r_df['created_utc'] = pd.to_datetime(r_df['created_utc'], utc=True, errors='coerce')
# ignore conditions with fewer than 100 samples
condition_counts_x = x_df['diagnosis'].value_counts()
valid_conditions = condition_counts_x[condition_counts_x >= 50].index
x_df = x_df[x_df['diagnosis'].isin(valid_conditions)]

condition_counts_r = r_df['diagnosis'].value_counts()
valid_conditions = condition_counts_r[condition_counts_r >= 50].index
r_df = r_df[r_df['diagnosis'].isin(valid_conditions)]

merged_df = pd.concat([x_df[cols], r_df[cols]], ignore_index=True)

merged_df.to_csv("data/metadata/majority-vote-combined.csv", index=False)

sample_df = (
    merged_df[merged_df['platform'] == 'twitter'].dropna(subset=['diagnosis'])
    .groupby('diagnosis', group_keys=False)
    .apply(lambda d: d.sample(n=min(30, len(d)), random_state=42).assign(diagnosis=d.name))
            .reset_index(drop=True)
)

# sample_df.to_csv("data/metadata/twitter-sample.csv", index=False)

/tmp/ipykernel_1708430/3197862020.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  x_df['created_utc'] = pd.to_datetime(x_df['created_utc'], utc=True, errors='coerce')


## prepare a reddit only dataset

In [6]:
import pandas as pd
from pathlib import Path

Path("data/metadata").mkdir(parents=True, exist_ok=True)

r = r.copy()
r.columns = r.columns.str.strip().str.lower()


r['is_comment'] = r['content_type'].fillna('').astype(str).str.lower().eq('comment')

r_df = r[['post_id', 'text', 'author', 'created_utc', 'subreddit', 'is_comment', 'diagnosis']].rename(columns={
    'post_id': 'id'
})

r_df['platform'] = 'reddit'

cols = ['id', 'text', 'author', 'created_utc', 'subreddit', 'is_comment', 'diagnosis', 'platform']

r_df['created_utc'] = pd.to_datetime(r_df['created_utc'], utc=True, errors='coerce')

condition_counts_r = r_df['diagnosis'].value_counts()
valid_conditions = condition_counts_r[condition_counts_r >= 50].index
r_df = r_df[r_df['diagnosis'].isin(valid_conditions)]


r_df.to_csv("data/metadata/majority-vote-reddit.csv", index=False)

sample_df = (
    r_df.dropna(subset=['diagnosis'])
    .groupby('diagnosis', group_keys=False)
    .apply(lambda d: d.sample(n=min(20, len(d)), random_state=42).assign(diagnosis=d.name))
            .reset_index(drop=True)
)

# sample_df.to_csv("data/metadata/reddit-sample.csv", index=False)

In [7]:
# pd.read_csv('~/Downloads/reddit-sample-nour - reddit-sample-nour.csv')

## llm as judge

In [ ]:
import os
import re
import json
import time
import pandas as pd
import torch
from tqdm.auto import tqdm
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_QWEN = "Qwen/Qwen2-72B"
MODEL_COL = "Qwen2-72B"
ALL_MODEL_COLS = [MODEL_COL]

CSV_PATH = "data/x.csv"
OUTPUT_PATH = "data/updated_dataset.csv"

MAX_NEW_TOKENS = 150
BATCH_SIZE = 16  # Adjust based on your GPU memory (2-8 for 72B model with 4-bit)

SYSTEM_PROMPT = """
You are a board-certified psychiatrist and clinical researcher specializing in mental health diagnosis. Your native language is Arabic, and you are fluent in English. You understand Arabic dialects — including Gulf, Egyptian, Levantine, Maghrebi, and Modern Standard Arabic (MSA) — as well as slang, religious expressions, sarcasm, figurative language, and informal social media communication. You are familiar with the cultural and social context of mental health in Arab societies.

Your task is text classification only. Classify based on what the author explicitly states — do not infer or assume a diagnosis.

TASK:
The post was collected using self-disclosure keyword phrases for {condition} ({condition_ar}).
Determine whether the post is:

* <genuine>: a real, first-person self-disclosure of {condition}, OR
* <not_confirmed>: NOT a real self-disclosure (false positive).

DEFAULT RULE:
When in doubt, label <not_confirmed>.

---

REASON TAGS (list ALL that apply)

Genuine evidence:
* explicit_self_disclosure: Clear first-person statement of having the condition
* treatment_or_medication: Mentions own medication, therapy, doctor visits, or treatment history
* symptom_description_self: Describes personal symptoms or lived experience related to the condition
* help_seeking_self: Asking for help, resources, or support for their own condition

Not genuine evidence:
* metaphor_or_exaggeration: Condition used figuratively, emotionally, or dramatically
* humor_or_sarcasm: Used jokingly or as a punchline
* third_person_reference: Talking about someone else's condition
* negation_or_denial: Explicitly states they do NOT have the condition
* self_questioning_or_uncertain: Expresses suspicion or asks without confirmed diagnosis
* professional_or_organizational_account: Bio indicates doctor, clinic, media, awareness page, etc.
* community_or_repost_account: Account reposts followers' messages (e.g., "من الخاص")
* news_or_public_content: News, educational, awareness, or public content not about the author

---

CLASSIFICATION RULES

Label <genuine> if the author:
* Sincerely states they have {condition}
* Describes their personal experience living with {condition}
* Seeks help, treatment, or support for their own {condition}
* Discusses their own medication, symptoms, or diagnosis history

Label <not_confirmed> if ANY of the following apply:
* Humor or sarcasm where the condition is used metaphorically or jokingly
* Figurative or metaphorical use of the condition
* News articles, celebrity quotes, or reporting someone else's condition
* Shared links where the phrase appears only in the title or external content
* Professional or educational posts about the condition
* Community or repost accounts sharing followers' messages
* Roleplay, fiction, literary, or promotional content
* Self-questioning or uncertainty without a confirmed diagnosis
* Explicit negation or denial of the condition
* Third-person discussion of someone else's condition
* Insulting or stigmatizing usage
* A vague or context-free mention with no sincere self-disclosure intent (e.g., used in a joking or throwaway manner — note: brevity alone does not make a post not_genuine; a short sincere statement is genuine)

---

USING THE BIO

* Use BOTH the post and the bio together to make the decision.
* If the bio indicates a professional, organizational, media, or community account → strongly supports <not_confirmed>.
* EXCEPTION: If the post contains a clear and explicit first-person statement about the author's own diagnosis or treatment (not a general statement), prioritize the post even if the bio suggests a professional role.
* If the bio indicates a patient identity (e.g., "مريض فصام", "schizophrenia patient") → supports <genuine>.
* A post addressed to a doctor, charity, or support organization supports <genuine>.
* Mentioning specific medications, symptoms, treatment history, or duration strongly supports <genuine>.

---

OUTPUT FORMAT
Output ONLY the following two fields. No preamble, reasoning, or explanation.

Label: <genuine> or <not_confirmed>
Reasons: one or more tags from the REASON TAGS list above, comma-separated
"""

USER_TEMPLATE = """Post: {post}
Bio: {bio}
Condition: {condition} ({condition_ar})

Label:
Reasons:"""

VALID_LABELS = {"<genuine>", "<not_confirmed>"}
VALID_REASONS = {
    "explicit_self_disclosure", "treatment_or_medication",
    "symptom_description_self", "help_seeking_self",
    "metaphor_or_exaggeration", "humor_or_sarcasm", "third_person_reference",
    "negation_or_denial", "self_questioning_or_uncertain",
    "professional_or_organizational_account", "community_or_repost_account",
    "news_or_public_content"
}

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_QWEN,
    trust_remote_code=True,
    use_fast=False
)

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_QWEN,
    device_map="auto",
    quantization_config=quant_config,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).eval()

def build_system_prompt(condition, condition_ar):
    return SYSTEM_PROMPT.format(condition=condition, condition_ar=condition_ar)

def parse_output(raw_text):
    if not raw_text or not raw_text.strip() or raw_text.startswith("API_ERROR"):
        return {"label": "PARSE_FAIL", "reasons": "PARSE_FAIL", "parse_status": "fail"}

    raw = raw_text.strip()
    label_match = re.search(r"Label\s*:\s*(<genuine>|<not_confirmed>)", raw, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else None

    if not label:
        if "<genuine>" in raw and "<not_confirmed>" not in raw:
            label = "<genuine>"
        elif "<not_confirmed>" in raw:
            label = "<not_confirmed>"
        else:
            label = "PARSE_FAIL"

    reasons_match = re.search(r"Reasons?\s*:\s*([a-z_, ]+)", raw, re.IGNORECASE)
    reasons = []

    if reasons_match:
        candidates = [r.strip().lower() for r in reasons_match.group(1).split(",")]
        reasons = [r for r in candidates if r in VALID_REASONS]

    if label in VALID_LABELS and reasons:
        status = "ok"
    elif label in VALID_LABELS:
        status = "partial"
        reasons = reasons if reasons else ["PARSE_FAIL"]
    else:
        status = "fail"
        label = "PARSE_FAIL"
        reasons = ["PARSE_FAIL"]

    return {"label": label, "reasons": ", ".join(reasons), "parse_status": status}

def apply_parser(df, model_col):
    parsed = df[model_col].apply(lambda x: parse_output(str(x)))
    df[f"{model_col}__label"] = parsed.apply(lambda x: x["label"])
    df[f"{model_col}__reasons"] = parsed.apply(lambda x: x["reasons"])
    df[f"{model_col}__status"] = parsed.apply(lambda x: x["parse_status"])
    return df

def get_todo_indices(df, model_col):
    if model_col not in df.columns:
        df[model_col] = None
        return list(df.index)

    status_col = f"{model_col}__status"

    if status_col in df.columns:
        done = df[status_col].notna() & df[status_col].eq("ok")
        has_error = df[model_col].astype(str).str.startswith("API_ERROR")
        done = done & ~has_error
    else:
        done = df[model_col].notna() & ~df[model_col].astype(str).str.startswith("API_ERROR")

    return df[~done].index.tolist()

def prepare_batch_messages(posts, bios, conditions, conditions_ar):
    """Prepare a batch of messages for the model"""
    batch_texts = []
    for post, bio, condition, condition_ar in zip(posts, bios, conditions, conditions_ar):
        messages = [
            {"role": "system", "content": build_system_prompt(condition, condition_ar)},
            {"role": "user", "content": USER_TEMPLATE.format(
                post=post,
                bio=bio,
                condition=condition,
                condition_ar=condition_ar
            )}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        batch_texts.append(text)
    return batch_texts

def call_qwen_batch(posts, bios, conditions, conditions_ar):
    """Process a batch of inputs"""
    batch_texts = prepare_batch_messages(posts, bios, conditions, conditions_ar)
    
    # Tokenize batch with padding
    inputs = tokenizer(
        batch_texts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True,
        max_length=4096  # Adjust based on model's max length
    ).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode each output, removing the input prompt
    generated_texts = []
    for i, output in enumerate(output_ids):
        # Get only the generated tokens (after input length)
        input_length = inputs["input_ids"][i].shape[0]
        generated_tokens = output[input_length:]
        generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        generated_texts.append(generated_text)
    
    return generated_texts

def run_qwen(df):
    if MODEL_COL not in df.columns:
        df[MODEL_COL] = None

    todo = get_todo_indices(df, MODEL_COL)
    
    # Process in batches
    for i in tqdm(range(0, len(todo), BATCH_SIZE), desc=f"Qwen2-72B (batch size={BATCH_SIZE})"):
        batch_indices = todo[i:i+BATCH_SIZE]
        batch_rows = df.loc[batch_indices]
        
        # Prepare batch data
        posts = [str(row.get("text", "")) for _, row in batch_rows.iterrows()]
        bios = [str(row.get("user_bio", "")) for _, row in batch_rows.iterrows()]
        conditions = [str(row.get("diagnosis", "schizophrenia")) for _, row in batch_rows.iterrows()]
        conditions_ar = [str(row.get("matched_keyword", "")) for _, row in batch_rows.iterrows()]
        
        try:
            # Process batch
            batch_results = call_qwen_batch(posts, bios, conditions, conditions_ar)
            
            # Store results
            for idx, result in zip(batch_indices, batch_results):
                df.at[idx, MODEL_COL] = result
                
        except Exception as e:
            # Handle batch error - mark all in batch as failed
            for idx in batch_indices:
                df.at[idx, MODEL_COL] = f"API_ERROR: {str(e)}"
        
        # Save intermediate results
        if (i // BATCH_SIZE + 1) % 10 == 0:  # Save every 10 batches
            df = apply_parser(df, MODEL_COL)
            save_results(df)
    
    return apply_parser(df, MODEL_COL)

def load_dataset():
    return pd.read_csv(CSV_PATH, encoding="utf-8-sig")

def save_results(df):
    df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

if __name__ == "__main__":
    df = load_dataset()
    # df = df[:5]  # Keep your test limit
    df = run_qwen(df)
    save_results(df)

Loading weights:   0%|          | 2/963 [00:02<17:50,  1.11s/it]/gpfs/automountdir/gpfs/homes/SEAS/home/g21775526/code/venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Qwen2-72B (batch size=16):  11%|█         | 104/944 [2:05:59<16:57:04, 72.65s/it][transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [9]:
import pandas as pd
df = pd.read_csv('data/updated_dataset.csv')
df["Qwen2-72B"].value_counts()

Qwen2-72B
Human: Label: <genuine>\nReasons: explicit_self-disclosure, symptom description self                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 141
Human: Label: <genuine>\nReasons: explicit_self-disclosure, symptom description self, help seeking self\n\nAssistant: